Given string s (digits only) and queries [l, r], for each query: take substring s[l..r], concatenate its non-zero digits into x (if none, x = 0), let sum = digit-sum of x, answer is x * sum, modulo 10⁹+7. Return one answer per query.

s = "10203004", queries = [[0,7],[1,3],[4,6]] → [12340, 4, 9]
s = "1000", queries = [[0,3],[1,1]] → [1, 0]
s = "9876543210", queries = [[0,9]] → [444444137] (that's 987654321 * 45 = 44444444445, reduced mod 10⁹+7)

In [ ]:
def solver(s, queries):
    output = []
    for query in queries:
        # print(query)
        if query[0] == query[1]:
            output.append(int(s[query[0]]) * int(s[query[0]]))
        elif query[0] < query[1]:
            selected_s = s[query[0]:query[1]+1]
            zero_removed_s = ''
            query_sum = 0
            for i in selected_s:
                if i != '0':
                    zero_removed_s += i
                    query_sum += int(i)
            # print("zero_removed_s", zero_removed_s)
            # print("query_sum", query_sum)
            if zero_removed_s != '':
                value_to_append = (int(zero_removed_s) * query_sum) % ((1000000000) + 7)
            else:
                value_to_append = 0
            output.append(value_to_append)
        else:
            output.append(0)
    return output

print(solver("10203004", [[0,7],[1,3],[4,6]]), "expected [12340, 4, 9]")
print(solver("1000", [[0,3],[1,1]]), "expected [1, 0]")
print(solver("9876543210", [[0,9]]), "expected [444444137]")
print(solver("00000", [[0,4],[1,2]]), "expected [0, 0]")
print(solver("12345", [[0,4],[1,3],[2,2]]), "expected [185175, 2928, 9]") # logically this test case is wrong. Program is producing correctly. Expectation is wrong. 

[12340, 4, 9] expected [12340, 4, 9]
[1, 0] expected [1, 0]
[444444137] expected [444444137]
[0, 0] expected [0, 0]
[185175, 2106, 9] expected [185175, 2928, 9]


In [23]:
def solver(s, queries):
    output = []
    for l, r in queries:
        selected_s = s[l:r+1]
        zero_removed_s = ''
        query_sum = 0
        for i in selected_s:
            if i != '0':
                zero_removed_s += i
                query_sum += int(i)
        if zero_removed_s != '':
            output.append((int(zero_removed_s) * query_sum) % (10**9 + 7))
        else:
            output.append(0)
    return output

print(solver("10203004", [[0,7],[1,3],[4,6]]), "expected [12340, 4, 9]")
print(solver("1000", [[0,3],[1,1]]), "expected [1, 0]")
print(solver("9876543210", [[0,9]]), "expected [444444137]")
print(solver("00000", [[0,4],[1,2]]), "expected [0, 0]")
print(solver("12345", [[0,4],[1,3],[2,2]]), "expected [185175, 2106(file says 2928, wrong), 9]")

[12340, 4, 9] expected [12340, 4, 9]
[1, 0] expected [1, 0]
[444444137] expected [444444137]
[0, 0] expected [0, 0]
[185175, 2106, 9] expected [185175, 2106(file says 2928, wrong), 9]


In [28]:
import time, random

MOD = 10**9 + 7

def solver_fast(s, queries):
    m = len(s)
    digit_sum_prefix = [0] * (m + 1)
    nz_count_prefix = [0] * (m + 1)
    horner_prefix = [0] * (m + 1)
    pow10 = [1] * (m + 1)

    for i in range(m):
        d = int(s[i])
        digit_sum_prefix[i+1] = digit_sum_prefix[i] + d
        pow10[i+1] = (pow10[i] * 10) % MOD
        if d != 0:
            nz_count_prefix[i+1] = nz_count_prefix[i] + 1
            horner_prefix[i+1] = (horner_prefix[i] * 10 + d) % MOD
        else:
            nz_count_prefix[i+1] = nz_count_prefix[i]
            horner_prefix[i+1] = horner_prefix[i]

    output = []
    for l, r in queries:
        total_sum = digit_sum_prefix[r+1] - digit_sum_prefix[l]
        k = nz_count_prefix[r+1] - nz_count_prefix[l]
        x_mod = (horner_prefix[r+1] - horner_prefix[l] * pow10[k]) % MOD
        output.append((x_mod * total_sum) % MOD)
    return output

# Correctness vs all 5 known cases
print(solver_fast("10203004", [[0,7],[1,3],[4,6]]), "expected [12340, 4, 9]")
print(solver_fast("1000", [[0,3],[1,1]]), "expected [1, 0]")
print(solver_fast("9876543210", [[0,9]]), "expected [444444137]")
print(solver_fast("00000", [[0,4],[1,2]]), "expected [0, 0]")
print(solver_fast("12345", [[0,4],[1,3],[2,2]]), "expected [185175, 2106, 9]")

# Scale test: does it actually survive 10^5 x 10^5 in reasonable time?
random.seed(0)
big_s = ''.join(random.choice("0123456789") for _ in range(100_000))
big_queries = []
for _ in range(100_000):
    a, b = random.randint(0, 99_999), random.randint(0, 99_999)
    big_queries.append([min(a,b), max(a,b)])

start = time.time()
print(big_s)
print(big_queries)
solver_fast(big_s, big_queries)
print(f"100k string / 100k queries: {time.time()-start:.2f}s")

start = time.time()
solver(big_s, big_queries)
print(f"100k string / 100k queries: {time.time()-start:.2f}s")

[12340, 4, 9] expected [12340, 4, 9]
[1, 0] expected [1, 0]
[444444137] expected [444444137]
[0, 0] expected [0, 0]
[185175, 2106, 9] expected [185175, 2106, 9]
...
[[25096, 38374], [23055, 53502], [86363, 87191], [47179, 95080], [75922, 82896], [42973, 61970], [32395, 99522], [64509, 89500], [20855, 40442], [67567, 73645], [16885, 86108], [27336, 74500], [22576, 38403], [31834, 87595], [19290, 35595], [21951, 80744], [78315, 97277], [9855, 91794], [19529, 36824], [2905, 63000], [6776, 34433], [15619, 53533], [25979, 48575], [40302, 56239], [90223, 91259], [40895, 94436], [33580, 35829], [40638, 50398], [12679, 74996], [8375, 92399], [53155, 74571], [21756, 76850], [48826, 62940], [26498, 57197], [29085, 49148], [35932, 63491], [23118, 85814], [66430, 93531], [8647, 48273], [18382, 47277], [18687, 56291], [49384, 82311], [33365, 61428], [29815, 52988], [7423, 92141], [80666, 98592], [4133, 56584], [15002, 61505], [8621, 16979], [3902, 42683], [43066, 58035], [17500, 72255], [19704, 889

ValueError: Exceeds the limit (4300 digits) for integer string conversion: value has 11947 digits; use sys.set_int_max_str_digits() to increase the limit